# FIUBench GA checkpoint effect diagnostic — Kaggle T4

공개 Stage-I 베이스 모델과 scheduler 수정 GA checkpoint를 동일한 Forget5 20문항에서 직접 비교합니다. 재학습과 Core 재평가는 하지 않습니다.

Kaggle 설정에서 **T4 GPU**와 **Internet ON**을 선택하고, scheduler 수정 학습 Notebook의 저장된 Output 하나만 Add Input으로 연결하세요. 셀 1 설치 후 커널 재시작 셀을 실행한 다음, 환경 확인 셀부터 아래로 순서대로 실행합니다.

검사 범위: 원본 checkpoint 경로와 SHA-256, 452개 tensor 완전 로드, 공개 베이스 대비 loss·정답 확률·생성·EM 변화, LoRA 병합 전후 loss 일치.


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
if [ ! -d FIUBench_kaggle ]; then
  git clone https://github.com/npngjjm/FIUBench_kaggle.git
fi
python -m pip uninstall -y torchao || true
python -m pip install --no-cache-dir --upgrade \
  'transformers==4.46.3' \
  'tokenizers==0.20.3' \
  'peft==0.13.2' \
  'accelerate==0.34.2' \
  'huggingface_hub==0.36.2' \
  'numpy==2.1.3' \
  hydra-core omegaconf datasets sentencepiece protobuf scipy pillow pyyaml
echo 'INSTALL_COMPLETE — 다음 셀에서 커널을 재시작하세요.'


설치 완료 후 아래 셀을 한 번 실행합니다. 연결이 돌아오면 이 셀은 건너뛰고 환경 확인 셀부터 진행하세요. Stop Session은 누르지 않습니다.


In [ ]:
import os
os._exit(0)


In [ ]:
import gc, hashlib, json, math, os, statistics, sys
from importlib.metadata import version
from pathlib import Path
import torch
from huggingface_hub import HfApi

repo = Path('/kaggle/working/FIUBench_kaggle')
assert repo.is_dir(), repo
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
expected = {
    'transformers': '4.46.3', 'tokenizers': '0.20.3', 'peft': '0.13.2',
    'accelerate': '0.34.2', 'huggingface-hub': '0.36.2', 'numpy': '2.1.3',
}
for package, wanted in expected.items():
    installed = version(package)
    print(package, installed)
    assert installed == wanted, (package, installed, wanted)
assert torch.cuda.is_available(), 'Kaggle에서 T4 GPU를 켜세요.'
print('torch', torch.__version__, 'GPU', torch.cuda.get_device_name(0))

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

candidates = [path for path in Path('/kaggle/input').rglob('checkpoint.pt') if not any(part.startswith('step_') for part in path.parts)]
print('최종 checkpoint 후보:', *map(str, candidates), sep='\n')
assert len(candidates) == 1, 'scheduler 수정 학습 Output 하나만 Add Input으로 연결하세요.'
source_checkpoint = candidates[0]
source_text = str(source_checkpoint).lower()
assert 'scheduler' in source_text and 'fast_scheduler_fix' in source_text, source_checkpoint
checkpoint_sha = sha256(source_checkpoint)
state_probe = torch.load(source_checkpoint, map_location='cpu', weights_only=True)
assert isinstance(state_probe, dict) and len(state_probe) == 452
assert all(torch.is_tensor(v) and torch.isfinite(v).all().item() for v in state_probe.values())
assert sum('lora_' in k for k in state_probe) == 448
assert sum('projector' in k for k in state_probe) == 4
del state_probe
gc.collect()
model_path = 'gray311/vlm_unlearning_ft_llava_phi_3_mini'
model_revision = HfApi().model_info(model_path).sha
clip_model = 'openai/clip-vit-large-patch14-336'
clip_revision = HfApi().model_info(clip_model).sha
print('checkpoint SHA256:', checkpoint_sha)
print('Stage-I revision:', model_revision)
print('CLIP revision:', clip_revision)
print('CHECKPOINT_PROVENANCE_OK')


In [ ]:
import shutil, zipfile
from collections import defaultdict
from huggingface_hub import hf_hub_download
from PIL import Image

split = json.loads((repo / 'dataset/split.json').read_text())
rows = [json.loads(line) for line in (repo / 'dataset/full.json').read_text().splitlines() if line.strip()][:400]
forget_ids = set(split['forget5'])
forget_rows = [row for row in rows if row['unique_id'] in forget_ids]
assert len(forget_rows) == 20
samples = []
for identity_index, row in enumerate(forget_rows):
    qa_index = identity_index % len(row['qa_list'])
    qa = row['qa_list'][qa_index]
    samples.append({
        'identity': row['unique_id'], 'qa_index': qa_index, 'image_path': row['image_path'],
        'question': qa['question'], 'answer': qa['answer'], 'keywords': qa['keywords'],
    })
assert len(samples) == 20
dataset_revision = HfApi().dataset_info('gray311/FIUBench').sha
archive = hf_hub_download(
    repo_id='gray311/FIUBench', filename='SFHQ.zip', repo_type='dataset',
    revision=dataset_revision, local_dir='/kaggle/working/fiu_diagnostic_download',
)
needed = {sample['image_path'] for sample in samples}
with zipfile.ZipFile(archive) as zipped:
    entries = defaultdict(list)
    for member in zipped.infolist():
        path = Path(member.filename)
        if not member.is_dir() and '__MACOSX' not in path.parts and not path.name.startswith('._'):
            entries[path.name].append(member)
    for image_path in needed:
        destination = (repo / image_path).resolve()
        assert destination.is_relative_to((repo / 'dataset').resolve())
        matches = entries[destination.name]
        assert len(matches) == 1, (destination.name, len(matches))
        destination.parent.mkdir(parents=True, exist_ok=True)
        with zipped.open(matches[0]) as source, destination.open('wb') as target:
            shutil.copyfileobj(source, target)
        with Image.open(destination) as image:
            image.verify()
for sample in samples:
    sample['image_path'] = str((repo / sample['image_path']).resolve())
    assert Path(sample['image_path']).is_file()
print('진단 표본:', len(samples), 'identities:', len({s['identity'] for s in samples}))
print('dataset revision:', dataset_revision)
print('DIAGNOSTIC_DATA_OK')


In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, CLIPImageProcessor, LlavaForConditionalGeneration
from data_module import preprocess_v1
from utils import get_model_identifiers_from_yaml

tokenizer = AutoTokenizer.from_pretrained(model_path, revision=model_revision, use_fast=True)
assert tokenizer.is_fast and tokenizer.__class__.__name__ == 'LlamaTokenizerFast'
tokenizer.padding_side = 'left'
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
image_processor = CLIPImageProcessor.from_pretrained(clip_model, revision=clip_revision)
model_cfg = get_model_identifiers_from_yaml('llava-phi')
question_tag, answer_tag = model_cfg['question_start_tag'], model_cfg['answer_tag']
roles = [question_tag, answer_tag]

def exact_match(prediction, keywords):
    score = 0.0
    for keyword in keywords:
        if keyword.lower() in prediction.lower():
            score += 1.0 / len(keywords)
    return min(1.0, score)

def evaluate_one(model, sample, generate=True):
    question, answer = sample['question'].capitalize(), sample['answer'].capitalize()
    conversation = question_tag + '<image>\n' + question + answer_tag + answer
    encoded = tokenizer(conversation, max_length=512, truncation=True, return_tensors='pt')
    labels = preprocess_v1(tokenizer, encoded['input_ids'], conversation, roles)
    image = image_processor.preprocess(Image.open(sample['image_path']), return_tensors='pt')['pixel_values']
    device = next(model.parameters()).device
    image = image.to(device=device, dtype=torch.float16)
    batch = {
        'input_ids': encoded['input_ids'].to(device), 'attention_mask': encoded['attention_mask'].to(device),
        'labels': labels.to(device), 'pixel_values': image,
    }
    with torch.inference_mode():
        output = model(**batch)
    loss = float(output.loss.float().cpu())
    del output
    prediction = ''
    if generate:
        decoded = tokenizer.batch_decode(encoded['input_ids'])[0]
        boundary = answer_tag.replace('\n', '')
        assert decoded.count(boundary) == 1, (boundary, decoded)
        prompt = decoded.split(boundary)[0].strip(' ') + boundary
        prompt_inputs = tokenizer.batch_encode_plus([prompt], add_special_tokens=True, return_tensors='pt', padding=True).to(device)
        with torch.inference_mode():
            generated = model.generate(
                input_ids=prompt_inputs.input_ids, attention_mask=prompt_inputs.attention_mask, pixel_values=image,
                max_new_tokens=50, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id,
            )
        prediction = tokenizer.batch_decode(generated[:, prompt_inputs.input_ids.shape[-1]:], skip_special_tokens=True)[0]
        prediction = prediction[:prediction.find('.') + 1]
    return {
        'loss': loss, 'answer_geomean_probability': math.exp(-loss), 'prediction': prediction,
        'exact_match': exact_match(prediction, sample['keywords']) if generate else None,
    }

def evaluate_set(model, label):
    model.eval()
    results = []
    for index, sample in enumerate(samples, 1):
        result = evaluate_one(model, sample, generate=True)
        results.append(result)
        print(f"{label} {index:02d}/20 loss={result['loss']:.6f} EM={result['exact_match']:.3f} pred={result['prediction'][:90]!r}")
    return results
print('TOKENIZER_AND_DIAGNOSTIC_HELPERS_OK')


In [ ]:
print('Loading the public Stage-I base model...')
base_model = LlavaForConditionalGeneration.from_pretrained(
    model_path, revision=model_revision, attn_implementation='sdpa', torch_dtype=torch.float16,
).cuda().eval()
base_results = evaluate_set(base_model, 'BASE')
print('BASE_EVALUATION_OK')


In [ ]:
target_modules = r'.*language_model.*\.(up_proj|k_proj|linear_2|down_proj|v_proj|q_proj|o_proj|gate_proj|linear_1)'
adapter_model = get_peft_model(base_model, LoraConfig(
    r=128, lora_alpha=256, target_modules=target_modules, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
))
state = torch.load(source_checkpoint, map_location='cpu', weights_only=True)
model_state = adapter_model.state_dict()
unknown = sorted(set(state) - set(model_state))
assert not unknown, unknown[:10]
load_result = adapter_model.load_state_dict(state, strict=False)
assert not load_result.unexpected_keys, load_result.unexpected_keys[:10]
loaded_state = adapter_model.state_dict()
mismatched = []
for key, expected_tensor in state.items():
    actual = loaded_state[key].detach().cpu().to(expected_tensor.dtype)
    if not torch.equal(actual, expected_tensor):
        mismatched.append(key)
assert not mismatched, mismatched[:10]
print('CHECKPOINT_ALL_452_TENSORS_LOADED')
del model_state, loaded_state, state
gc.collect()
adapter_model.eval()
unmerged_losses = [evaluate_one(adapter_model, sample, generate=False)['loss'] for sample in samples[:3]]
merged_model = adapter_model.merge_and_unload().eval()
merged_losses = [evaluate_one(merged_model, sample, generate=False)['loss'] for sample in samples[:3]]
merge_loss_deltas = [abs(a - b) for a, b in zip(unmerged_losses, merged_losses)]
merge_equivalence_ok = max(merge_loss_deltas) < 0.02
print('LoRA merge loss deltas:', merge_loss_deltas)
assert merge_equivalence_ok, merge_loss_deltas
print('LORA_MERGE_LOSS_EQUIVALENCE_OK')
ga_results = evaluate_set(merged_model, 'GA')
print('GA_EVALUATION_OK')


In [ ]:
loss_deltas = [ga['loss'] - base['loss'] for base, ga in zip(base_results, ga_results)]
changed_predictions = sum(base['prediction'] != ga['prediction'] for base, ga in zip(base_results, ga_results))
checkpoint_has_effect = max(abs(delta) for delta in loss_deltas) > 1e-5
summary = {
    'source_checkpoint': str(source_checkpoint), 'checkpoint_sha256': checkpoint_sha,
    'stage1_model_revision': model_revision, 'dataset_revision': dataset_revision, 'samples': len(samples),
    'base_mean_loss': statistics.fmean(item['loss'] for item in base_results),
    'ga_mean_loss': statistics.fmean(item['loss'] for item in ga_results),
    'mean_loss_increase_ga_minus_base': statistics.fmean(loss_deltas),
    'base_mean_answer_probability': statistics.fmean(item['answer_geomean_probability'] for item in base_results),
    'ga_mean_answer_probability': statistics.fmean(item['answer_geomean_probability'] for item in ga_results),
    'base_mean_em': statistics.fmean(item['exact_match'] for item in base_results),
    'ga_mean_em': statistics.fmean(item['exact_match'] for item in ga_results),
    'changed_predictions': changed_predictions, 'merge_loss_deltas_first_three': merge_loss_deltas,
    'merge_equivalence_ok': merge_equivalence_ok,
    'checkpoint_has_measurable_loss_effect': checkpoint_has_effect,
}
details = [{'sample': sample, 'base': base, 'ga': ga} for sample, base, ga in zip(samples, base_results, ga_results)]
output_path = Path(f'/kaggle/working/ga_checkpoint_effect_diagnostic_{checkpoint_sha[:12]}.json')
output_path.write_text(json.dumps({'summary': summary, 'details': details}, indent=2, ensure_ascii=False))
print('\n' + '=' * 72)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('saved:', output_path)
assert checkpoint_has_effect, 'Checkpoint has no measurable loss effect'
print('GA_CHECKPOINT_EFFECT_DIAGNOSTIC_OK')


마지막 JSON 요약과 `GA_CHECKPOINT_EFFECT_DIAGNOSTIC_OK`를 확인하세요. `/kaggle/working/ga_checkpoint_effect_diagnostic_<hash>.json`을 보존하려면 Save Version → Quick Save를 실행합니다.
